# Comparative survival analysis: TCGA GI cancers (COAD, READ, STAD)

Kernel 1 (the `thesis` env: lifelines / scipy<1.12). Preprocessing, EDA,
Kaplan-Meier and Cox live here. The Random Survival Forest / SHAP / transfer
run in the separate `thesis_rsf` env and read the clean CSVs this script writes.

EXPANDED CLINICAL FEATURE SET
-----------------------------
The models used to run on three predictors (age, sex, overall stage). Per the
supervisor ("use all the variables we can"), the clean per-patient files now
carry the full set of usable clinical variables, and Cox is fitted on the
expanded set. The pipeline itself is unchanged - EDA and KM still summarise the
core variables; only the feature set fed to the models grows.

What goes where, and why:
  - Cox  : age, sex, overall stage, prior malignancy, residual disease,
           treatment received, race, ethnicity  (one-hot, complete-case on
           age+stage, ridge-penalised).
  - Forest/SHAP (other env): the above PLUS T, N and M stage (granular staging
           is collinear with overall stage, so it is kept out of Cox but is fine
           for a tree model). The clean CSV stores everything so both halves read
           the same definitions.

Excluded after inspecting the actual data:
  - classification_of_tumor : at the primary-diagnosis row this equals the
    primary-disease flag (constant 'primary'); its only variation comes from
    metastasis/recurrence rows, i.e. post-baseline progression -> leakage.
  - prior_treatment : >99% 'No' in every cohort (100% 'No' in STAD), near-zero
    variance. Stored in the CSV for transparency but not fed to the models.
  - vital_status : this is the outcome. Never a feature.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from scipy.stats import chi2

In [2]:
# ---- Paths ----
RAW_PATHS   = {'coad': 'Coad/clinical.tsv', 'read': 'Read/clinical.tsv', 'stad': 'Stad/clinical.tsv'}
CLEAN_DIR   = 'Clean'
FIGURE_DIR  = 'Figures'
RESULTS_DIR = 'Results'
for d in (CLEAN_DIR, FIGURE_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# ---- Labels and the house colour scheme ----
LABELS  = {'coad': 'COAD (Colon)', 'read': 'READ (Rectum)', 'stad': 'STAD (Stomach)'}
COLOURS = {'coad': '#1E6091', 'read': '#2A9D8F', 'stad': '#E63946'}

STAGE_ORDER   = ['Stage I', 'Stage II', 'Stage III', 'Stage IV']
STAGE_COLOURS = {'Stage I': '#2c7fb8', 'Stage II': '#41b6c4', 'Stage III': '#fdae61', 'Stage IV': '#d7191c'}
GENDER_ORDER   = ['Male', 'Female']
GENDER_COLOURS = {'Male': '#1E6091', 'Female': '#E9967A'}
AGE_ORDER   = ['<50', '50-59', '60-69', '70+']
AGE_COLOURS = ['#2c7fb8', '#41b6c4', '#fdae61', '#d7191c']

In [3]:
# ---- GDC column names used out of the ~200 in the clinical file ----
ID_COL       = 'cases.case_id'
PRIMARY_FLAG = 'diagnoses.diagnosis_is_primary_disease'
VITAL_COL    = 'demographic.vital_status'
DEATH_COL    = 'demographic.days_to_death'
FOLLOWUP_COL = 'diagnoses.days_to_last_follow_up'
AGE_COL      = 'demographic.age_at_index'
GENDER_COL   = 'demographic.gender'
RACE_COL     = 'demographic.race'
ETH_COL      = 'demographic.ethnicity'
STAGE_COL    = 'diagnoses.ajcc_pathologic_stage'
T_COL        = 'diagnoses.ajcc_pathologic_t'
N_COL        = 'diagnoses.ajcc_pathologic_n'
M_COL        = 'diagnoses.ajcc_pathologic_m'
PRIORMAL_COL = 'diagnoses.prior_malignancy'
PRIORTX_COL  = 'diagnoses.prior_treatment'
RESID_COL    = 'diagnoses.residual_disease'    # diagnoses-level (~64% populated); the
                                               # treatments-level copy is ~0.5% populated, unusable.
TREAT_COL    = 'treatments.treatment_or_therapy'

# ---- Value collapse maps (built from the real value sets in the three cohorts) ----
# Overall stage: TCGA sub-stages -> four main groups.
STAGE_MAP = {
    'Stage I':   'Stage I',   'Stage IA':  'Stage I',   'Stage IB':  'Stage I',
    'Stage II':  'Stage II',  'Stage IIA': 'Stage II',  'Stage IIB': 'Stage II',  'Stage IIC': 'Stage II',
    'Stage III': 'Stage III', 'Stage IIIA':'Stage III', 'Stage IIIB':'Stage III', 'Stage IIIC':'Stage III',
    'Stage IV':  'Stage IV',  'Stage IVA': 'Stage IV',  'Stage IVB': 'Stage IV',  'Stage IVC': 'Stage IV',
}

def map_t(v):
    """T1..T4; TX / Tis / missing -> Unknown."""
    if pd.isna(v):            return 'Unknown'
    v = str(v)
    for k in ('T1', 'T2', 'T3', 'T4'):
        if v.startswith(k):   return k
    return 'Unknown'

def map_n(v):
    """N0..N3; NX / missing -> Unknown."""
    if pd.isna(v):            return 'Unknown'
    v = str(v)
    for k in ('N0', 'N1', 'N2', 'N3'):
        if v.startswith(k):   return k
    return 'Unknown'

def map_m(v):
    """M0 / M1; MX / missing -> Unknown."""
    if pd.isna(v):            return 'Unknown'
    v = str(v)
    if v.startswith('M0'):    return 'M0'
    if v.startswith('M1'):    return 'M1'
    return 'Unknown'

def map_yes_no(v):
    """Yes / No; 'not reported' / missing -> Unknown."""
    if pd.isna(v):            return 'Unknown'
    v = str(v).strip().lower()
    if v == 'yes':            return 'Yes'
    if v == 'no':             return 'No'
    return 'Unknown'

def map_residual(v):
    """R0 vs R+ (residual present, R1 or R2); RX / missing -> Unknown.
    R1 alone is very sparse (4 / 2 / 15 patients), so R1 and R2 are pooled."""
    if pd.isna(v):            return 'Unknown'
    v = str(v).strip().upper()
    if v == 'R0':             return 'R0'
    if v in ('R1', 'R2'):     return 'R+'
    return 'Unknown'

def map_race(v):
    """White / Black / Asian / Other-Unknown (American Indian and Pacific
    Islander are too few to stand alone, so they join 'not reported')."""
    if pd.isna(v):            return 'Other/Unknown'
    v = str(v).strip().lower()
    if v == 'white':                       return 'White'
    if v == 'black or african american':   return 'Black'
    if v == 'asian':                       return 'Asian'
    return 'Other/Unknown'

def map_ethnicity(v):
    if pd.isna(v):            return 'Unknown'
    v = str(v).strip().lower()
    if v == 'hispanic or latino':      return 'Hispanic'
    if v == 'not hispanic or latino':  return 'Non-Hispanic'
    return 'Unknown'

In [4]:
# ---- Cox feature specification ----
# Each categorical: (reference level, non-reference levels in display order).
# A dummy is created for every non-reference level; constant dummies (a level that
# is absent or fills the whole cohort) are dropped automatically before fitting.
COX_CATEGORICALS = {
    'stage':             ('Stage I',      ['Stage II', 'Stage III', 'Stage IV']),
    'prior_malignancy':  ('No',           ['Yes', 'Unknown']),
    'residual_disease':  ('R0',           ['R+', 'Unknown']),
    'treatment_therapy': ('No',           ['Yes', 'Unknown']),
    'race':              ('White',        ['Black', 'Asian', 'Other/Unknown']),
    'ethnicity':         ('Non-Hispanic', ['Hispanic', 'Unknown']),
}
# T/N/M are deliberately NOT in Cox (collinear with overall stage) - they go to the
# forest/SHAP instead. prior_treatment is excluded (near-zero variance).

# dummy column name = "<Feature>: <level>", except stage keeps the bare label.
def cox_colname(feat, level):
    return level if feat == 'stage' else f'{COX_FEATURE_LABELS[feat]}: {level}'

COX_FEATURE_LABELS = {
    'prior_malignancy':  'Prior malignancy',
    'residual_disease':  'Residual disease',
    'treatment_therapy': 'Treatment received',
    'race':              'Race',
    'ethnicity':         'Ethnicity',
}
# friendly names for the two continuous / binary terms; dummy columns are self-labelling
COVAR_LABELS = {'age': 'Age (per year)', 'gender_male': 'Male vs Female'}

# canonical covariate order for the forest plot (only those present per cohort are shown)
def canonical_cox_order():
    order = ['age', 'gender_male']
    for feat, (_ref, levels) in COX_CATEGORICALS.items():
        order += [cox_colname(feat, lev) for lev in levels]
    return order

PENALIZERS = (0.1, 0.5, 1.0)   # ridge; escalates only if a fit fails to converge

# columns written to the clean per-patient CSVs (feed the forest/SHAP/transfer scripts)
FINAL_COLUMNS = ['case_id', 'survival_time', 'event', 'vital_status', 'age', 'gender',
                 'stage', 'stage_original', 't_stage', 'n_stage', 'm_stage',
                 'prior_malignancy', 'prior_treatment', 'residual_disease',
                 'treatment_therapy', 'race', 'ethnicity']

In [5]:
# ---- Plot style ----
plt.rcParams['figure.dpi']        = 150
plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize']    = 13
plt.rcParams['axes.labelsize']    = 11
plt.rcParams['xtick.labelsize']   = 10
plt.rcParams['ytick.labelsize']   = 10

try:                       # only exists on newer pandas; harmless to skip otherwise
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

## Helper functions (defined once, used throughout)

In [6]:
def load(key):
    """Read one cleaned per-patient file back in. Used by the later sections."""
    return pd.read_csv(f'{CLEAN_DIR}/{key}_clean.csv')


def replace_tcga_missing(df):
    """TCGA writes missing values as "'--"; turn those into real NaN."""
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"'--": np.nan, "--": np.nan, "nan": np.nan, "": np.nan})
    return df


def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan


def primary_value(grp, col):
    """Value from a primary-diagnosis row if one carries it, else from any row.
    This is how stage was always taken; the same rule now feeds T/N/M and the
    other diagnosis-level fields, which also vary across a patient's rows."""
    primary_rows = grp[grp['_primary'] == 'true']
    source = primary_rows if (col in primary_rows and primary_rows[col].notna().any()) else grp
    return first_non_null(source[col])


def collapse_to_one_row_per_patient(df):
    """Group all of a patient's rows into a single summary row.

    Demographics are constant within a patient; follow-up uses the latest contact;
    stage / T / N / M and the other diagnosis-level fields come from the primary
    diagnosis; treatment is aggregated across all of the patient's treatment rows.
    Raw values are stored here and mapped to clean categories in the next step."""
    df = df.copy()
    df['_days_to_death']    = pd.to_numeric(df[DEATH_COL],    errors='coerce')
    df['_days_to_followup'] = pd.to_numeric(df[FOLLOWUP_COL], errors='coerce')
    df['_age']              = pd.to_numeric(df[AGE_COL],      errors='coerce')
    df['_primary']          = df[PRIMARY_FLAG].astype(str).str.strip().str.lower()

    records = []
    for case_id, grp in df.groupby(ID_COL):
        # treatment received: any 'yes' -> Yes, else any 'no' -> No, else Unknown
        treat_vals = grp[TREAT_COL].dropna().astype(str).str.strip().str.lower().tolist()
        if any(v == 'yes' for v in treat_vals):
            treatment = 'Yes'
        elif any(v == 'no' for v in treat_vals):
            treatment = 'No'
        else:
            treatment = 'Unknown'

        records.append({
            'case_id':                'patient_' + str(case_id)[:8],
            'vital_status':           first_non_null(grp[VITAL_COL]),
            'days_to_death':          grp['_days_to_death'].max(),
            'days_to_last_follow_up': grp['_days_to_followup'].max(),   # latest contact
            'age':                    first_non_null(grp['_age']),
            'gender':                 first_non_null(grp[GENDER_COL]),
            'race_raw':               first_non_null(grp[RACE_COL]),
            'ethnicity_raw':          first_non_null(grp[ETH_COL]),
            'stage_original':         primary_value(grp, STAGE_COL),
            't_raw':                  primary_value(grp, T_COL),
            'n_raw':                  primary_value(grp, N_COL),
            'm_raw':                  primary_value(grp, M_COL),
            'prior_malignancy_raw':   primary_value(grp, PRIORMAL_COL),
            'prior_treatment_raw':    primary_value(grp, PRIORTX_COL),
            'residual_raw':           primary_value(grp, RESID_COL),
            'treatment_therapy':      treatment,     # already aggregated (cross-row)
        })
    return pd.DataFrame(records)


def build_survival_and_clean(df):
    """Build the event flag and survival time, then map every raw field to its
    clean modelling category."""
    df['event'] = (df['vital_status'].str.lower() == 'dead').astype(int)  # 1 = died
    df['survival_time'] = np.where(df['event'] == 1, df['days_to_death'], df['days_to_last_follow_up'])

    df['gender'] = df['gender'].str.strip().str.lower().replace({'male': 'Male', 'female': 'Female'})
    df['stage']  = df['stage_original'].map(STAGE_MAP)

    df['t_stage']          = df['t_raw'].apply(map_t)
    df['n_stage']          = df['n_raw'].apply(map_n)
    df['m_stage']          = df['m_raw'].apply(map_m)
    df['prior_malignancy'] = df['prior_malignancy_raw'].apply(map_yes_no)
    df['prior_treatment']  = df['prior_treatment_raw'].apply(map_yes_no)
    df['residual_disease'] = df['residual_raw'].apply(map_residual)
    df['race']             = df['race_raw'].apply(map_race)
    df['ethnicity']        = df['ethnicity_raw'].apply(map_ethnicity)
    return df


def process_cohort(path, name):
    """Read one cohort, collapse to one row per patient, drop unusable rows."""
    raw = pd.read_csv(path, sep='\t', low_memory=False)
    n_rows, n_cols, n_unique = raw.shape[0], raw.shape[1], raw[ID_COL].nunique()

    clean = build_survival_and_clean(collapse_to_one_row_per_patient(replace_tcga_missing(raw)))

    before = len(clean)
    clean = clean.dropna(subset=['survival_time'])
    clean = clean[clean['survival_time'] > 0].copy()
    dropped = before - len(clean)

    print(f"\n{'-'*54}\n {name}\n{'-'*54}")
    print(f" Raw rows in file:        {n_rows}  ({n_cols} columns)")
    print(f" Unique patients:         {n_unique}")
    print(f" Dropped (no valid time): {dropped} ({dropped/n_unique*100:.1f}% of patients)")
    print(f" Final analysable cohort: {len(clean)}")
    return clean


def add_age_group(df):
    df = df.copy()
    df['age_group'] = pd.cut(df['age'], bins=[0, 50, 60, 70, 200], labels=AGE_ORDER, right=False)
    return df


def median_or_nan(kmf):
    m = kmf.median_survival_time_
    return m if np.isfinite(m) else np.nan


def run_logrank(data, group_col):
    """Pairwise for two groups, multivariate for more than two."""
    groups = data[group_col].dropna().unique()
    if len(groups) < 2:
        return np.nan
    if len(groups) == 2:
        a = data[data[group_col] == groups[0]]
        b = data[data[group_col] == groups[1]]
        res = logrank_test(a['survival_time'], b['survival_time'],
                           event_observed_A=a['event'], event_observed_B=b['event'])
        return res.p_value
    return multivariate_logrank_test(data['survival_time'], data[group_col], data['event']).p_value


def build_cox_design(df):
    """Cox design matrix on the expanded clinical feature set.

    Complete-case on age + overall stage (unchanged from the 3-feature pipeline, so
    the analysable/Cox N is the same). Every other categorical carries an 'Unknown'
    level rather than dropping patients. Constant dummies - a level absent from, or
    filling, a cohort - are removed so lifelines does not choke on a zero-variance
    column. Returns the design and the list of dropped columns."""
    d = df.dropna(subset=['stage', 'age']).copy()

    X = pd.DataFrame(index=d.index)
    X['survival_time'] = d['survival_time'].astype(float)
    X['event']         = d['event'].astype(int)
    X['age']           = d['age'].astype(float)
    X['gender_male']   = (d['gender'] == 'Male').astype(int)

    for feat, (_ref, levels) in COX_CATEGORICALS.items():
        for lev in levels:
            X[cox_colname(feat, lev)] = (d[feat] == lev).astype(int)

    covars  = [c for c in X.columns if c not in ('survival_time', 'event')]
    dropped = [c for c in covars if X[c].nunique() < 2]     # constant -> no information
    X = X.drop(columns=dropped)
    return X, dropped


def fit_cox(design, penalizers=PENALIZERS):
    """Fit a ridge-penalised Cox model, escalating the penalty only if a fit fails
    to converge (the expanded design is wide relative to the events, especially in
    READ). Returns the fitted model and the penalizer that worked."""
    last_error = None
    for pen in penalizers:
        try:
            cph = CoxPHFitter(penalizer=pen)
            cph.fit(design, duration_col='survival_time', event_col='event')
            return cph, pen
        except Exception as e:      # convergence / linalg issues on sparse folds
            last_error = e
    raise last_error

## 1. Preprocessing
Collapse each cohort to one row per patient and save the clean files. The clean
CSVs now carry the full clinical feature set (including T/N/M for the forest).

In [7]:
print("Preprocessing (one row per patient)")
clean_frames = {key: process_cohort(path, LABELS[key]) for key, path in RAW_PATHS.items()}

# save the clean per-patient files (these also feed the Random Forest / SHAP scripts)
for key, df in clean_frames.items():
    df[[c for c in FINAL_COLUMNS if c in df.columns]].to_csv(f'{CLEAN_DIR}/{key}_clean.csv', index=False)
    print(f"Saved: {CLEAN_DIR}/{key}_clean.csv")

Preprocessing (one row per patient)

------------------------------------------------------
 COAD (Colon)
------------------------------------------------------
 Raw rows in file:        1801  (201 columns)
 Unique patients:         461
 Dropped (no valid time): 24 (5.2% of patients)
 Final analysable cohort: 437

------------------------------------------------------
 READ (Rectum)
------------------------------------------------------
 Raw rows in file:        642  (201 columns)
 Unique patients:         172
 Dropped (no valid time): 11 (6.4% of patients)
 Final analysable cohort: 161

------------------------------------------------------
 STAD (Stomach)
------------------------------------------------------
 Raw rows in file:        1595  (201 columns)
 Unique patients:         443
 Dropped (no valid time): 31 (7.0% of patients)
 Final analysable cohort: 412
Saved: Clean/coad_clean.csv
Saved: Clean/read_clean.csv
Saved: Clean/stad_clean.csv


In [8]:
# Add the age group column and use these clean frames for the rest of the analysis
datasets = {key: add_age_group(df) for key, df in clean_frames.items()}
coad, read, stad = datasets['coad'], datasets['read'], datasets['stad']

# core summary (unchanged), then the expanded categorical distributions
CATS_TO_REPORT = ['stage', 't_stage', 'n_stage', 'm_stage', 'prior_malignancy',
                  'prior_treatment', 'residual_disease', 'treatment_therapy', 'race', 'ethnicity']
for key, df in datasets.items():
    print(f"\n{LABELS[key]}  ({len(df)} patients)")
    print(f"  Deaths (event=1):  {df['event'].sum()} ({df['event'].mean()*100:.1f}%)")
    print(f"  Censored:          {(df['event']==0).sum()} ({(df['event']==0).mean()*100:.1f}%)")
    print(f"  Obs. time (days):  mean {df['survival_time'].mean():.1f}, "
          f"median {df['survival_time'].median():.0f}, max {df['survival_time'].max():.0f}")
    print(f"  Age:               mean {df['age'].mean():.1f}, median {df['age'].median():.0f}")
    print(f"  Gender:            {df['gender'].value_counts().to_dict()}")
    print("  Expanded clinical features (all patients):")
    for col in CATS_TO_REPORT:
        print(f"    {col:<18} {df[col].value_counts(dropna=False).to_dict()}")


COAD (Colon)  (437 patients)
  Deaths (event=1):  95 (21.7%)
  Censored:          342 (78.3%)
  Obs. time (days):  mean 870.2, median 672, max 4502
  Age:               mean 66.3, median 68
  Gender:            {'Male': 236, 'Female': 201}
  Expanded clinical features (all patients):
    stage              {'Stage II': 167, 'Stage III': 124, 'Stage I': 74, 'Stage IV': 61, nan: 11}
    t_stage            {'T3': 299, 'T2': 76, 'T4': 50, 'T1': 11, 'Unknown': 1}
    n_stage            {'N0': 257, 'N1': 103, 'N2': 77}
    m_stage            {'M0': 324, 'M1': 61, 'Unknown': 52}
    prior_malignancy   {'No': 382, 'Yes': 54, 'Unknown': 1}
    prior_treatment    {'No': 434, 'Yes': 3}
    residual_disease   {'R0': 314, 'Unknown': 97, 'R+': 26}
    treatment_therapy  {'No': 207, 'Yes': 188, 'Unknown': 42}
    race               {'White': 211, 'Other/Unknown': 159, 'Black': 56, 'Asian': 11}
    ethnicity          {'Non-Hispanic': 265, 'Unknown': 168, 'Hispanic': 4}

READ (Rectum)  (161 patients)


## 2. Exploratory data analysis
The observation-time histogram (Figure 1) mixes deaths and censored follow-up,
so its median is the *observation* median, not the survival median. The three
cohorts also have very similar ages, which matters for the cross-cancer comparison.

In [9]:
# Figure 1: observation-time distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 1: Observation Time Distributions by Cancer Type', fontsize=14, fontweight='bold', y=1.02)
for ax, (key, df) in zip(axes, datasets.items()):
    ax.hist(df['survival_time'], bins=40, color=COLOURS[key], alpha=0.8, edgecolor='white')
    ax.axvline(df['survival_time'].median(), color='black', linestyle='--', linewidth=1.5,
               label=f"Median obs.: {df['survival_time'].median():.0f} days")
    ax.set_title(LABELS[key]); ax.set_xlabel('Observation time (days)'); ax.set_ylabel('Number of patients')
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f'n = {len(df)}', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='#555555')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig1_survival_time_distributions.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig1_survival_time_distributions.png")

Saved: fig1_survival_time_distributions.png


In [10]:
# Figure 2: age distributions
fig, ax = plt.subplots(figsize=(10, 5))
for key, df in datasets.items():
    ax.hist(df['age'].dropna(), bins=30, alpha=0.6, color=COLOURS[key], label=LABELS[key], edgecolor='white')
ax.set_title('Figure 2: Age Distribution by Cancer Type', fontweight='bold')
ax.set_xlabel('Age at diagnosis (years)'); ax.set_ylabel('Number of patients'); ax.legend()
for i, (key, df) in enumerate(datasets.items()):
    ax.text(0.02, 0.92 - i * 0.07, f"{LABELS[key]} median age: {df['age'].median():.0f}",
            transform=ax.transAxes, fontsize=9, color=COLOURS[key])
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig2_age_distributions.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig2_age_distributions.png")

Saved: fig2_age_distributions.png


In [11]:
# Figure 3: gender breakdown
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Figure 3: Gender Distribution by Cancer Type', fontsize=14, fontweight='bold')
for ax, (key, df) in zip(axes, datasets.items()):
    counts = df['gender'].value_counts()
    bars = ax.bar(counts.index, counts.values, color=[COLOURS[key], '#BBBBBB'], edgecolor='white', width=0.5)
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{count}\n({count / len(df) * 100:.0f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(LABELS[key]); ax.set_xlabel('Gender'); ax.set_ylabel('Number of patients')
    ax.set_ylim(0, max(counts.values) * 1.2)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig3_gender_distribution.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig3_gender_distribution.png")

Saved: fig3_gender_distribution.png


In [12]:
# Figure 4: stage distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 4: Cancer Stage Distribution', fontsize=14, fontweight='bold')
for ax, (key, df) in zip(axes, datasets.items()):
    stage_counts = df['stage'].value_counts().reindex(STAGE_ORDER, fill_value=0)
    x = np.arange(len(STAGE_ORDER))
    bars = ax.bar(x, stage_counts.values, color=COLOURS[key], alpha=0.85, edgecolor='white')
    staged_total = df['stage'].notna().sum()
    for bar, count in zip(bars, stage_counts.values):
        pct = count / staged_total * 100 if staged_total else 0
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{count}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=8)
    n_missing = df['stage'].isna().sum()
    ax.text(0.97, 0.97, f'Missing: {n_missing} ({n_missing/len(df)*100:.0f}%)',
            transform=ax.transAxes, ha='right', va='top', fontsize=8, color='grey')
    ax.set_title(LABELS[key]); ax.set_xlabel('Stage'); ax.set_ylabel('Number of patients')
    ax.set_xticks(x); ax.set_xticklabels(STAGE_ORDER, rotation=15)
    ax.set_ylim(0, max(stage_counts.values) * 1.25)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig4_stage_distribution.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig4_stage_distribution.png")

Saved: fig4_stage_distribution.png


In [13]:
# Figure 5: death rate by cancer
death_rates = [df['event'].mean() * 100 for df in datasets.values()]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([LABELS[k] for k in datasets], death_rates, color=[COLOURS[k] for k in datasets],
              edgecolor='white', width=0.5)
for bar, rate in zip(bars, death_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Figure 5: Death Rate (% of patients who died) by Cancer Type', fontweight='bold')
ax.set_ylabel('Death rate (%)'); ax.set_ylim(0, max(death_rates) * 1.2)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig5_death_rate_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig5_death_rate_comparison.png")

Saved: fig5_death_rate_comparison.png


In [14]:
# Figure 6: summary table
def col(df):
    return [f"{len(df):,}", f"{df['event'].sum()}", f"{df['event'].mean()*100:.1f}%",
            f"{(df['event']==0).sum()}", f"{df['survival_time'].median():.0f}",
            f"{df['age'].mean():.1f}", f"{df['stage'].isna().mean()*100:.1f}%"]
summary_df = pd.DataFrame({
    'Metric': ['Total Patients', 'Deaths (n)', 'Death Rate (%)', 'Censored (n)',
               'Median Obs. Time (days)', 'Mean Age (years)', 'Stage Missing (%)'],
    'COAD (Colon)': col(coad), 'READ (Rectum)': col(read), 'STAD (Stomach)': col(stad),
})
fig, ax = plt.subplots(figsize=(11, 4)); ax.axis('off')
table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.8)
for j in range(len(summary_df.columns)):
    table[0, j].set_facecolor('#0D1B2A'); table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(summary_df) + 1):
    table[i, 0].set_facecolor('#F2F2F2'); table[i, 0].set_text_props(fontweight='bold')
    for j in range(1, len(summary_df.columns)):
        if i % 2 == 0:
            table[i, j].set_facecolor('#FAFAFA')
ax.set_title('Figure 6: Summary Statistics - All Three Cancer Datasets', fontweight='bold', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig6_summary_table.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig6_summary_table.png")

Saved: fig6_summary_table.png


## 3. Kaplan-Meier
Each cancer on its own. Overall survival, then split by stage, gender and age.

In [15]:
# overall median survival (censoring-aware, unlike the EDA observation median)
print("Overall median survival (Kaplan-Meier):")
for key, df in datasets.items():
    kmf = KaplanMeierFitter()
    kmf.fit(df['survival_time'], event_observed=df['event'])
    med = median_or_nan(kmf)
    print(f"  {LABELS[key]}: median survival = " + (f"{med:.0f} days" if np.isfinite(med) else "not reached"))

Overall median survival (Kaplan-Meier):
  COAD (Colon): median survival = 2821 days
  READ (Rectum): median survival = 1741 days
  STAD (Stomach): median survival = 881 days


In [16]:
# Figure 7: overall survival curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Figure 7: Overall Kaplan-Meier Survival Curves by Cancer Type', fontsize=14, fontweight='bold', y=1.02)
for ax, (key, df) in zip(axes, datasets.items()):
    kmf = KaplanMeierFitter()
    kmf.fit(df['survival_time'], event_observed=df['event'], label=LABELS[key])
    kmf.plot_survival_function(ax=ax, ci_show=True, color=COLOURS[key])
    med = median_or_nan(kmf)
    if np.isfinite(med):
        ax.axvline(med, color='black', linestyle='--', linewidth=1.2, label=f'Median: {med:.0f} days')
    ax.set_title(LABELS[key]); ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability')
    ax.set_ylim(0, 1.02); ax.legend(fontsize=9, loc='upper right')
    ax.text(0.97, 0.55, f'n = {len(df)}', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='#555555')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig7_km_overall.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig7_km_overall.png")

Saved: fig7_km_overall.png


In [17]:
# Figure 8: survival by stage (the wobbly tails are just where few patients remain at risk)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Figure 8: Kaplan-Meier Survival by Stage', fontsize=14, fontweight='bold', y=1.02)
for ax, (key, df) in zip(axes, datasets.items()):
    data = df.dropna(subset=['stage'])
    for stage in STAGE_ORDER:
        mask = data['stage'] == stage
        if mask.sum() == 0:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(data.loc[mask, 'survival_time'], event_observed=data.loc[mask, 'event'],
                label=f'{stage} (n={int(mask.sum())}, d={int(data.loc[mask, "event"].sum())})')
        kmf.plot_survival_function(ax=ax, ci_show=False, color=STAGE_COLOURS[stage])
    pval = run_logrank(data, 'stage')
    print(f"  {LABELS[key]} - log-rank by stage: p = {pval:.4g}")
    ax.set_title(f"{LABELS[key]}\nLog-rank p = {pval:.3g}")
    ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability'); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig8_km_by_stage.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig8_km_by_stage.png")

  COAD (Colon) - log-rank by stage: p = 1.268e-12
  READ (Rectum) - log-rank by stage: p = 0.0007613
  STAD (Stomach) - log-rank by stage: p = 5.924e-06
Saved: fig8_km_by_stage.png


In [18]:
# Figure 9: survival by gender
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Figure 9: Kaplan-Meier Survival by Gender', fontsize=14, fontweight='bold', y=1.02)
for ax, (key, df) in zip(axes, datasets.items()):
    data = df.dropna(subset=['gender'])
    for g in GENDER_ORDER:
        mask = data['gender'] == g
        if mask.sum() == 0:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(data.loc[mask, 'survival_time'], event_observed=data.loc[mask, 'event'],
                label=f'{g} (n={int(mask.sum())}, d={int(data.loc[mask, "event"].sum())})')
        kmf.plot_survival_function(ax=ax, ci_show=False, color=GENDER_COLOURS[g])
    pval = run_logrank(data, 'gender')
    print(f"  {LABELS[key]} - log-rank by gender: p = {pval:.4g}")
    ax.set_title(f"{LABELS[key]}\nLog-rank p = {pval:.3g}")
    ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability'); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig9_km_by_gender.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig9_km_by_gender.png")

  COAD (Colon) - log-rank by gender: p = 0.5472
  READ (Rectum) - log-rank by gender: p = 0.6774
  STAD (Stomach) - log-rank by gender: p = 0.6389
Saved: fig9_km_by_gender.png


In [19]:
# Figure 10: survival by age group
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Figure 10: Kaplan-Meier Survival by Age Group', fontsize=14, fontweight='bold', y=1.02)
for ax, (key, df) in zip(axes, datasets.items()):
    data = df.dropna(subset=['age_group'])
    for ag, colour in zip(AGE_ORDER, AGE_COLOURS):
        mask = data['age_group'] == ag
        if mask.sum() == 0:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(data.loc[mask, 'survival_time'], event_observed=data.loc[mask, 'event'],
                label=f'{ag} (n={int(mask.sum())}, d={int(data.loc[mask, "event"].sum())})')
        kmf.plot_survival_function(ax=ax, ci_show=False, color=colour)
    pval = run_logrank(data, 'age_group')
    print(f"  {LABELS[key]} - log-rank by age group: p = {pval:.4g}")
    ax.set_title(f"{LABELS[key]}\nLog-rank p = {pval:.3g}")
    ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability'); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig10_km_by_age_group.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig10_km_by_age_group.png")

  COAD (Colon) - log-rank by age group: p = 0.07456
  READ (Rectum) - log-rank by age group: p = 0.00129
  STAD (Stomach) - log-rank by age group: p = 0.2792
Saved: fig10_km_by_age_group.png


## 4. Cox proportional hazards (expanded clinical feature set)
Each covariate's effect with the others held constant. Complete-case on
age + stage (so n matches the KM analysable cohort minus stage-missing:
426 / 153 / 393). The design is ridge-penalised because it is wide relative to
the number of events, especially in READ; constant dummies are dropped per cohort.

In [20]:
models     = {}
model_pen  = {}
for key, df in datasets.items():
    design, dropped = build_cox_design(df)
    cph, pen = fit_cox(design)
    models[key], model_pen[key] = cph, pen

    print(f"\n{'-'*60}\n {LABELS[key]}   (n={len(design)}, deaths={int(design['event'].sum())}, "
          f"covariates={design.shape[1]-2}, penalizer={pen})\n{'-'*60}")
    if dropped:
        print(f"  Dropped constant dummies (level absent/constant in this cohort): {dropped}")
    cph.print_summary(columns=['coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p'], decimals=3)
    print(f"  Age HR per 10 years  : {np.exp(cph.params_['age'] * 10):.2f}")
    print(f"  Concordance (C-index): {cph.concordance_index_:.3f}")
    lr = cph.log_likelihood_ratio_test()
    print(f"  Overall model LR test: chi2={lr.test_statistic:.2f}, p={lr.p_value:.3g}")

    tidy = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    tidy.columns = ['HR', 'CI_lower_95', 'CI_upper_95', 'p_value']
    tidy.index = [COVAR_LABELS.get(i, i) for i in tidy.index]
    tidy.to_csv(f'{RESULTS_DIR}/cox_summary_{key}.csv')
    print(f"  Saved: {RESULTS_DIR}/cox_summary_{key}.csv")

# C-index summary across cohorts (single source of truth for the forest-script comparison)
cindex_df = pd.DataFrame([{'cohort': LABELS[k],
                           'Cox_C_index': round(models[k].concordance_index_, 3),
                           'n': int(build_cox_design(datasets[k])[0].shape[0]),
                           'penalizer': model_pen[k]} for k in datasets])
cindex_df.to_csv(f'{RESULTS_DIR}/cox_cindex_summary.csv', index=False)
print(f"\nSaved: {RESULTS_DIR}/cox_cindex_summary.csv")


------------------------------------------------------------
 COAD (Colon)   (n=426, deaths=90, covariates=16, penalizer=0.1)
------------------------------------------------------------


<lifelines.CoxPHFitter: fitted with 426 total observations, 336 right-censored observations>
             duration col = 'survival_time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 426
number of events observed = 90
   partial log-likelihood = -453.117
         time fit was run = 2026-07-12 22:24:18 UTC

---
                              coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%       p
covariate                                                                                   
age                          0.015     1.015               1.000               1.030   0.045
gender_male                 -0.014     0.986               0.695               1.399   0.939
Stage II                    -0.245     0.783               0.512               1.197   0.258
Stage III                    0.316     1.372               0.884               2.130   0.159
Stage IV                     1.157     3.179               1.900               5.318 <0.0005
Prior malignancy: Yes        0.100     1.105               0.668               1.829   0.696
Prior malignancy: Unknown   -0.995     0.370               0.005              30.121   0.658
Residual disease: R+         0.646     1.908               1.018               3.576   0.044
Residual disease: Unknown    0.393     1.482               0.962               2.283   0.075
Treatment received: Yes     -0.156     0.856               0.575               1.274   0.442
Treatment received: Unknown  0.324     1.383               0.769               2.489   0.279
Race: Black                  0.057     1.059               0.620               1.809   0.834
Race: Asian                  0.425     1.530               0.441               5.305   0.502
Race: Other/Unknown         -0.102     0.903               0.553               1.474   0.683
Ethnicity: Hispanic          1.151     3.162               0.728              13.740   0.125
Ethnicity: Unknown           0.059     1.061               0.665               1.691   0.805
---
Concordance = 0.762
Partial AIC = 938.233
log-likelihood ratio test = 49.327 on 16 df
-log2(p) of ll-ratio test = 15.058

  Age HR per 10 years  : 1.16
  Concordance (C-index): 0.762
  Overall model LR test: chi2=49.33, p=2.93e-05
  Saved: Results/cox_summary_coad.csv

------------------------------------------------------------
 READ (Rectum)   (n=153, deaths=24, covariates=15, penalizer=0.1)
------------------------------------------------------------
  Dropped constant dummies (level absent/constant in this cohort): ['Ethnicity: Hispanic']


<lifelines.CoxPHFitter: fitted with 153 total observations, 129 right-censored observations>
             duration col = 'survival_time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 153
number of events observed = 24
   partial log-likelihood = -86.439
         time fit was run = 2026-07-12 22:24:18 UTC

---
                              coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%     p
covariate                                                                                 
age                          0.034     1.034               1.002               1.068 0.038
gender_male                 -0.238     0.789               0.408               1.523 0.479
Stage II                    -0.232     0.793               0.356               1.766 0.571
Stage III                    0.175     1.192               0.548               2.590 0.658
Stage IV                     0.940     2.559               1.044               6.273 0.040
Prior malignancy: Yes       -0.596     0.551               0.154               1.974 0.360
Prior malignancy: Unknown   -0.746     0.474               0.016              13.749 0.664
Residual disease: R+         0.984     2.676               0.903               7.931 0.076
Residual disease: Unknown   -0.529     0.589               0.233               1.487 0.263
Treatment received: Yes     -0.472     0.624               0.316               1.231 0.173
Treatment received: Unknown  0.402     1.494               0.531               4.204 0.447
Race: Black                 -0.060     0.942               0.186               4.755 0.942
Race: Asian                 -0.103     0.902               0.002             330.291 0.973
Race: Other/Unknown         -0.302     0.739               0.328               1.665 0.465
Ethnicity: Unknown          -0.134     0.874               0.395               1.936 0.740
---
Concordance = 0.827
Partial AIC = 202.878
log-likelihood ratio test = 22.725 on 15 df
-log2(p) of ll-ratio test = 3.471

  Age HR per 10 years  : 1.40
  Concordance (C-index): 0.827
  Overall model LR test: chi2=22.73, p=0.0902
  Saved: Results/cox_summary_read.csv

------------------------------------------------------------
 STAD (Stomach)   (n=393, deaths=157, covariates=15, penalizer=0.1)
------------------------------------------------------------
  Dropped constant dummies (level absent/constant in this cohort): ['Prior malignancy: Unknown']


<lifelines.CoxPHFitter: fitted with 393 total observations, 236 right-censored observations>
             duration col = 'survival_time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 393
number of events observed = 157
   partial log-likelihood = -802.074
         time fit was run = 2026-07-12 22:24:18 UTC

---
                              coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%       p
covariate                                                                                   
age                          0.013     1.013               0.998               1.027   0.083
gender_male                  0.037     1.037               0.769               1.398   0.810
Stage II                    -0.092     0.912               0.615               1.353   0.648
Stage III                    0.420     1.521               1.059               2.186   0.023
Stage IV                     1.046     2.847               1.672               4.847 <0.0005
Prior malignancy: Yes       -0.350     0.705               0.262               1.893   0.488
Residual disease: R+         0.703     2.020               1.245               3.275   0.004
Residual disease: Unknown    1.193     3.296               2.030               5.350 <0.0005
Treatment received: Yes     -0.396     0.673               0.490               0.924   0.014
Treatment received: Unknown  0.097     1.102               0.701               1.731   0.673
Race: Black                  0.023     1.023               0.509               2.057   0.949
Race: Asian                 -0.148     0.863               0.580               1.283   0.465
Race: Other/Unknown         -0.204     0.816               0.503               1.324   0.410
Ethnicity: Hispanic         -2.255     0.105               0.021               0.513   0.005
Ethnicity: Unknown           0.116     1.122               0.776               1.624   0.540
---
Concordance = 0.700
Partial AIC = 1634.148
log-likelihood ratio test = 67.164 on 15 df
-log2(p) of ll-ratio test = 26.067

  Age HR per 10 years  : 1.14
  Concordance (C-index): 0.700
  Overall model LR test: chi2=67.16, p=1.42e-08
  Saved: Results/cox_summary_stad.csv

Saved: Results/cox_cindex_summary.csv


In [21]:
# proportional hazards check (small p flags a covariate whose effect changes over time)
for key, df in datasets.items():
    print(f"\n{'-'*40}\n {LABELS[key]}\n{'-'*40}")
    design, _ = build_cox_design(df)
    ph = proportional_hazard_test(models[key], design, time_transform='rank')
    ph.print_summary(decimals=3)


----------------------------------------
 COAD (Colon)
----------------------------------------


<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 426 total observations, 336 right-censored observations>
         test_name = proportional_hazard_test

---
                             test_statistic    p  -log2(p)
Ethnicity: Hispanic                    1.04 0.31      1.70
Ethnicity: Unknown                     0.04 0.84      0.25
Prior malignancy: Unknown              0.02 0.88      0.19
Prior malignancy: Yes                  1.13 0.29      1.80
Race: Asian                            0.27 0.61      0.72
Race: Black                            0.24 0.63      0.67
Race: Other/Unknown                    0.27 0.61      0.72
Residual disease: R+                   0.01 0.90      0.15
Residual disease: Unknown              0.60 0.44      1.20
Stage II                               0.53 0.47      1.10
Stage III                              0.43 0.51      0.97
Stage IV                               1.73 0.19      2.40
Treatment received: Unknown            6.94 0.01      6.89
Treatment received: Yes                5.18 0.02      5.45
age                                    0.14 0.71      0.50
gender_male                            0.43 0.51      0.96


----------------------------------------
 READ (Rectum)
----------------------------------------


<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 153 total observations, 129 right-censored observations>
         test_name = proportional_hazard_test

---
                             test_statistic    p  -log2(p)
Ethnicity: Unknown                     0.05 0.83      0.27
Prior malignancy: Unknown              0.02 0.89      0.18
Prior malignancy: Yes                  0.25 0.62      0.70
Race: Asian                            0.00 0.98      0.03
Race: Black                            1.80 0.18      2.48
Race: Other/Unknown                    0.11 0.74      0.44
Residual disease: R+                   0.21 0.65      0.63
Residual disease: Unknown              0.56 0.45      1.14
Stage II                               1.08 0.30      1.74
Stage III                              0.01 0.94      0.09
Stage IV                               2.04 0.15      2.71
Treatment received: Unknown            0.07 0.78      0.35
Treatment received: Yes                1.12 0.29      1.79
age                                    1.90 0.17      2.57
gender_male                            0.01 0.93      0.11


----------------------------------------
 STAD (Stomach)
----------------------------------------


<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 393 total observations, 236 right-censored observations>
         test_name = proportional_hazard_test

---
                             test_statistic      p  -log2(p)
Ethnicity: Hispanic                    0.10   0.75      0.42
Ethnicity: Unknown                     0.73   0.39      1.35
Prior malignancy: Yes                  0.47   0.49      1.02
Race: Asian                            1.59   0.21      2.27
Race: Black                            0.36   0.55      0.86
Race: Other/Unknown                    0.41   0.52      0.93
Residual disease: R+                   0.88   0.35      1.52
Residual disease: Unknown              0.09   0.77      0.38
Stage II                               0.13   0.72      0.47
Stage III                              0.03   0.86      0.22
Stage IV                               0.65   0.42      1.26
Treatment received: Unknown            0.11   0.74      0.44
Treatment received: Yes               11.95 <0.005     10.84
age                                    0.72   0.39      1.34
gender_male                            0.01   0.90      0.14

In [22]:
# Figure 11: hazard ratio forest plots (each cohort shows its own surviving covariates)
CANON = canonical_cox_order()
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle('Figure 11: Cox Hazard Ratios by Cancer Type (expanded clinical model)',
             fontsize=17, fontweight='bold', y=1.02)
for ax, (key, cph) in zip(axes, models.items()):
    covars = [c for c in CANON if c in cph.summary.index]
    s = cph.summary.reindex(covars)
    hr    = s['exp(coef)'].values
    lower = s['exp(coef) lower 95%'].values
    upper = s['exp(coef) upper 95%'].values
    y_pos = np.arange(len(covars))
    ax.errorbar(hr, y_pos, xerr=[hr - lower, upper - hr], fmt='o', color=COLOURS[key],
                ecolor=COLOURS[key], elinewidth=1.5, capsize=3, markersize=6)
    ax.axvline(1.0, color='grey', linestyle='--', linewidth=1)
    ax.set_xscale('log')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([COVAR_LABELS.get(c, c) for c in covars], fontsize=12)
    ax.invert_yaxis()
    ax.set_title(f"{LABELS[key]}\nC-index = {cph.concordance_index_:.3f}  (penalizer {model_pen[key]})")
    ax.set_xlabel('Hazard ratio (log scale)')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig11_cox_forest.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig11_cox_forest.png")



fig.set_size_inches(20, 9)          # was cramped; give the rows room
fig.savefig('Figures/fig11_cox_hr.png', dpi=300, bbox_inches='tight')

Saved: fig11_cox_forest.png


In [23]:
# Figure 12: stage hazard ratio gradient across cancers (trend, not absolute survival)
x_labels = ['I (ref)', 'II', 'III', 'IV']
x_pos = np.arange(len(x_labels))
fig, ax = plt.subplots(figsize=(9, 5.5))
for key, cph in models.items():
    s = cph.summary['exp(coef)']
    hrs = [1.0] + [s.get(f'Stage {r}', np.nan) for r in ['II', 'III', 'IV']]
    ax.plot(x_pos, hrs, marker='o', linewidth=2, markersize=8, color=COLOURS[key], label=LABELS[key])
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
ax.set_yscale('log'); ax.set_xticks(x_pos); ax.set_xticklabels(x_labels)
ax.set_xlabel('Stage'); ax.set_ylabel('Hazard ratio vs Stage I (log scale)')
ax.set_title('Figure 12: Stage Hazard Ratio Gradient Across Cancers', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig12_stage_hr_gradient.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig12_stage_hr_gradient.png")

print("\nAll done. Figures in Figures/, Cox tables in Results/.")

Saved: fig12_stage_hr_gradient.png

All done. Figures in Figures/, Cox tables in Results/.


## 5. Adjusted between-cancer comparison

Supervisor question 1: does colon cancer have worse survival than stomach cancer
after adjusting for age, stage and sex? Unadjusted, stomach is clearly worse (death
rate 40.8% vs 21.7%, median survival 881 vs 2,821 days). But stomach is also
diagnosed at a later stage, so the question is whether the difference survives
adjustment for stage, age and sex.

This pools the two cohorts into a single Cox model with a cancer-type indicator. The
indicator's hazard ratio is the adjusted between-cancer effect: it compares colon
with stomach with age, stage and sex held constant. Because "colon" could mean colon
only (COAD) or colorectal (COAD + READ), both are run.

Pooling assumes the age, stage and sex effects are shared across the cancers; this is
the standard adjusted comparison, and the proportional-hazards check flags whether the
cancer effect is constant over time (it may not be, if stomach's excess risk is early).

In [24]:
# Reference level for stage is Stage I; reload the clean per-patient files.
STAGE_LEVELS = ['Stage II', 'Stage III', 'Stage IV']

coad, read, stad = load('coad'), load('read'), load('stad')


def run_pooled(title, colon_frames, colon_name):
    """Pool the colon-type cohort(s) with stomach and fit an adjusted Cox model.
    The indicator is 1 for the colon-type group and 0 for stomach (stomach = reference),
    so its hazard ratio reads directly as 'colon-type vs stomach'."""
    colon = pd.concat(colon_frames, ignore_index=True).copy(); colon['is_colon'] = 1
    stom  = stad.copy(); stom['is_colon'] = 0
    pool  = pd.concat([colon, stom], ignore_index=True)

    d = pool.dropna(subset=['age', 'stage']).copy()          # complete-case on age + stage
    X = pd.DataFrame(index=d.index)
    X['survival_time'] = d['survival_time'].astype(float)
    X['event']         = d['event'].astype(int)
    X['age']           = d['age'].astype(float)
    X['male']          = (d['gender'] == 'Male').astype(int)
    for s in STAGE_LEVELS:
        X[s] = (d['stage'] == s).astype(int)
    X[f'{colon_name}_vs_stomach'] = d['is_colon']

    n_colon = int((d['is_colon'] == 1).sum()); n_stom = int((d['is_colon'] == 0).sum())
    print(f"\n{'='*66}\n {title}\n{'='*66}")
    print(f"  {colon_name}: n={n_colon} (deaths {int(X.loc[d['is_colon']==1,'event'].sum())}); "
          f"stomach: n={n_stom} (deaths {int(X.loc[d['is_colon']==0,'event'].sum())})")

    cph = CoxPHFitter().fit(X, duration_col='survival_time', event_col='event')
    cph.print_summary(columns=['coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p'], decimals=3)

    term = f'{colon_name}_vs_stomach'
    hr = cph.summary.loc[term, 'exp(coef)']
    lo = cph.summary.loc[term, 'exp(coef) lower 95%']
    hi = cph.summary.loc[term, 'exp(coef) upper 95%']
    p  = cph.summary.loc[term, 'p']
    print(f"\n  Adjusted {colon_name} vs stomach: HR = {hr:.2f} (95% CI {lo:.2f} to {hi:.2f}), p = {p:.3g}")
    if p < 0.05 and hr < 1:
        verdict = (f"After adjustment {colon_name} has LOWER risk than stomach "
                   f"(about {1/hr:.1f}x lower), so {colon_name} does NOT have worse survival.")
    elif p < 0.05 and hr > 1:
        verdict = f"After adjustment {colon_name} has HIGHER risk than stomach."
    else:
        verdict = f"After adjustment there is no significant survival difference between {colon_name} and stomach."
    print(f"  -> {verdict}")
    print(f"  Concordance (C-index): {cph.concordance_index_:.3f}")

    # is the cancer effect constant over time?
    ph = proportional_hazard_test(cph, X, time_transform='rank')
    ph_p = ph.summary.loc[term, 'p'] if term in ph.summary.index else np.nan
    if np.isfinite(ph_p):
        note = "constant over time" if ph_p > 0.05 else "time-varying (single HR is a summary; gap likely concentrated early)"
        print(f"  Proportional-hazards check on the cancer term: p = {ph_p:.3g} ({note}).")

    tidy = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    tidy.columns = ['HR', 'CI_low', 'CI_high', 'p']
    tidy.to_csv(f'{RESULTS_DIR}/q1_pooled_cox_{colon_name}.csv')
    return hr, lo, hi, p

In [25]:
print("Supervisor Q1: colon-type vs stomach survival, adjusted for age, stage and sex")
run_pooled("Colon (COAD) vs Stomach (STAD)",        [coad],        'colon')
run_pooled("Colorectal (COAD+READ) vs Stomach (STAD)", [coad, read], 'colorectal')

print(f"\nSaved per-model summaries to {RESULTS_DIR}/q1_pooled_cox_*.csv")

Supervisor Q1: colon-type vs stomach survival, adjusted for age, stage and sex

 Colon (COAD) vs Stomach (STAD)
  colon: n=426 (deaths 90); stomach: n=393 (deaths 157)


<lifelines.CoxPHFitter: fitted with 819 total observations, 572 right-censored observations>
             duration col = 'survival_time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 819
number of events observed = 247
   partial log-likelihood = -1424.140
         time fit was run = 2026-07-12 22:24:21 UTC

---
                   coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%       p
covariate                                                                        
age               0.029     1.030               1.018               1.042 <0.0005
male             -0.036     0.965               0.744               1.252   0.787
Stage II          0.540     1.716               0.990               2.974   0.054
Stage III         1.143     3.135               1.853               5.305 <0.0005
Stage IV          2.057     7.825               4.451              13.756 <0.0005
colon_vs_stomach -1.023     0.360               0.274               0.472 <0.0005
---
Concordance = 0.723
Partial AIC = 2860.280
log-likelihood ratio test = 144.544 on 6 df
-log2(p) of ll-ratio test = 92.876


  Adjusted colon vs stomach: HR = 0.36 (95% CI 0.27 to 0.47), p = 1.79e-13
  -> After adjustment colon has LOWER risk than stomach (about 2.8x lower), so colon does NOT have worse survival.
  Concordance (C-index): 0.723
  Proportional-hazards check on the cancer term: p = 0.766 (constant over time).

 Colorectal (COAD+READ) vs Stomach (STAD)
  colorectal: n=579 (deaths 114); stomach: n=393 (deaths 157)


<lifelines.CoxPHFitter: fitted with 972 total observations, 701 right-censored observations>
             duration col = 'survival_time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 972
number of events observed = 271
   partial log-likelihood = -1591.403
         time fit was run = 2026-07-12 22:24:21 UTC

---
                        coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%       p
covariate                                                                             
age                    0.034     1.035               1.023               1.047 <0.0005
male                  -0.083     0.920               0.718               1.179   0.511
Stage II               0.508     1.662               0.987               2.800   0.056
Stage III              1.134     3.107               1.890               5.107 <0.0005
Stage IV               2.064     7.877               4.642              13.366 <0.0005
colorectal_vs_stomach -1.112     0.329               0.255               0.424 <0.0005
---
Concordance = 0.733
Partial AIC = 3194.806
log-likelihood ratio test = 180.817 on 6 df
-log2(p) of ll-ratio test = 118.403


  Adjusted colorectal vs stomach: HR = 0.33 (95% CI 0.26 to 0.42), p = 7.03e-18
  -> After adjustment colorectal has LOWER risk than stomach (about 3.0x lower), so colorectal does NOT have worse survival.
  Concordance (C-index): 0.733
  Proportional-hazards check on the cancer term: p = 0.634 (constant over time).

Saved per-model summaries to Results/q1_pooled_cox_*.csv


## 6. Invariant prediction across cancers

Supervisor question 3: which predictors maintain the same relationship with survival
across both cancers? This is the idea behind Invariant Causal Prediction: a predictor
whose effect on survival is stable across environments (here, colorectal vs stomach)
behaves like a shared, possibly causal, prognostic factor, whereas a predictor whose
effect changes between the cancers is context-specific.

The analysis fits each covariate's effect separately in colorectal (COAD + READ) and
in stomach (STAD), then formally tests whether that effect differs between the two by
adding a covariate-by-cancer interaction to a pooled Cox model and doing a likelihood
-ratio test. A non-significant interaction means the covariate's effect is invariant
across the two cancers.

Two environments (colorectal, stomach) are used rather than three cohorts, which
matches the "both cancers" framing and avoids an unstable split on the small rectal
cohort. Covariates tested: age, sex, stage (ordinal I..IV) and residual disease.

In [26]:
# Ordinal stage, the four shared covariates, and their labels for the figure.
STAGE_ORD = {'Stage I': 1, 'Stage II': 2, 'Stage III': 3, 'Stage IV': 4}
COVARS    = ['age', 'male', 'stage_ord', 'residual_present']
LABELS    = {'age': 'Age (per year)', 'male': 'Male vs Female',
             'stage_ord': 'Stage (per level)', 'residual_present': 'Residual disease present'}

def make_design(df):
    """Complete-case design with the four shared covariates (no event/time yet)."""
    d = df.dropna(subset=['age', 'stage']).copy()
    X = pd.DataFrame(index=d.index)
    X['survival_time']    = d['survival_time'].astype(float)
    X['event']            = d['event'].astype(int)
    X['age']              = d['age'].astype(float)
    X['male']             = (d['gender'] == 'Male').astype(int)
    X['stage_ord']        = d['stage'].map(STAGE_ORD).astype(float)
    X['residual_present'] = (d['residual_disease'] == 'R+').astype(int)
    return X

coad, read, stad = load('coad'), load('read'), load('stad')
crc  = make_design(pd.concat([coad, read], ignore_index=True))   # environment 0: colorectal
stom = make_design(stad)                                         # environment 1: stomach

## Per-environment covariate effects

In [27]:
def fit_hrs(X, name):
    cph = CoxPHFitter().fit(X, duration_col='survival_time', event_col='event')
    print(f"\n{name}: n={len(X)}, deaths={int(X['event'].sum())}, C-index={cph.concordance_index_:.3f}")
    return cph

cph_crc  = fit_hrs(crc,  'Colorectal (COAD+READ)')
cph_stom = fit_hrs(stom, 'Stomach (STAD)')


Colorectal (COAD+READ): n=579, deaths=114, C-index=0.737

Stomach (STAD): n=393, deaths=157, C-index=0.644


## Invariance test: does each covariate's effect differ between the cancers?
Pooled Cox with a stomach indicator; for each covariate, a likelihood-ratio test
compares the model with and without that covariate's interaction with the indicator.

In [28]:
crc_p  = crc.copy();  crc_p['stom']  = 0
stom_p = stom.copy(); stom_p['stom'] = 1
pool = pd.concat([crc_p, stom_p], ignore_index=True)

base_cols = COVARS + ['stom']
def fit_ll(cols):
    cph = CoxPHFitter().fit(pool[['survival_time', 'event'] + cols],
                            duration_col='survival_time', event_col='event')
    return cph.log_likelihood_

ll_base = fit_ll(base_cols)

rows = []
for c in COVARS:
    inter = f'{c}_x_stom'
    pool[inter] = pool[c] * pool['stom']
    ll_full = fit_ll(base_cols + [inter])
    lr = 2 * (ll_full - ll_base)
    p_int = chi2.sf(lr, df=1)
    hr_crc  = float(np.exp(cph_crc.params_[c]))
    hr_stom = float(np.exp(cph_stom.params_[c]))
    rows.append({'covariate': LABELS[c],
                 'HR_colorectal': round(hr_crc, 3),
                 'HR_stomach': round(hr_stom, 3),
                 'interaction_p': round(float(p_int), 4),
                 'invariant': 'yes' if p_int > 0.05 else 'no (cancer-specific)'})

inv = pd.DataFrame(rows)
print("\n" + "=" * 74)
print("INVARIANCE OF COVARIATE EFFECTS ACROSS COLORECTAL vs STOMACH")
print("=" * 74)
print(inv.to_string(index=False))
print("\nInvariant predictors (effect stable across both cancers) behave as shared")
print("prognostic factors; a significant interaction means the effect is cancer-specific.")
inv.to_csv(f'{RESULTS_DIR}/q3_invariance.csv', index=False)
print(f"Saved: {RESULTS_DIR}/q3_invariance.csv")


INVARIANCE OF COVARIATE EFFECTS ACROSS COLORECTAL vs STOMACH
               covariate  HR_colorectal  HR_stomach  interaction_p invariant
          Age (per year)          1.040       1.024         0.4578       yes
          Male vs Female          0.882       1.010         0.8146       yes
       Stage (per level)          2.225       1.659         0.1029       yes
Residual disease present          1.599       1.738         0.4784       yes

Invariant predictors (effect stable across both cancers) behave as shared
prognostic factors; a significant interaction means the effect is cancer-specific.
Saved: Results/q3_invariance.csv


## Figure: covariate effect in each cancer, side by side

In [29]:
fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(COVARS))[::-1]
for cph, name, colour, off in [(cph_crc, 'Colorectal', '#1E6091', 0.13),
                               (cph_stom, 'Stomach', '#E63946', -0.13)]:
    hr    = np.array([np.exp(cph.params_[c]) for c in COVARS])
    lower = np.array([np.exp(cph.confidence_intervals_.loc[c].iloc[0]) for c in COVARS])
    upper = np.array([np.exp(cph.confidence_intervals_.loc[c].iloc[1]) for c in COVARS])
    ax.errorbar(hr, y + off, xerr=[hr - lower, upper - hr], fmt='o', color=colour,
                ecolor=colour, elinewidth=1.5, capsize=3, markersize=6, label=name)
ax.axvline(1.0, color='grey', linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_yticks(y); ax.set_yticklabels([LABELS[c] for c in COVARS])
ax.set_xlabel('Hazard ratio (log scale)')
ax.set_title('Figure (Q3): Covariate effects across colorectal and stomach\n'
             'overlapping estimates indicate an invariant relationship', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_invariance_q3.png', bbox_inches='tight', dpi=150)
plt.close()
print(f"Saved: {FIGURE_DIR}/fig_invariance_q3.png")

Saved: Figures/fig_invariance_q3.png
